# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrathibhaShaliniS/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Distributions of every field this audit will test — impressions, clicks, CTR, position (excluding the "no data" zeros), engagement rate, scroll rate. Traffic metrics are almost always heavy-tailed: a few giant pages, a long tail of tiny ones. Checking this first decides how every later test has to be done (weighted rates and medians, not plain averages).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/PrathibhaShaliniS/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

cols = ["impressions_90d", "clicks_90d", "ctr", "engagement_rate", "scroll_rate"]
stats = df[cols].agg(["mean", "median", "max"]).T
pos = df.loc[df["avg_position"] > 0, "avg_position"]
stats.loc["avg_position (pos>0)"] = [pos.mean(), pos.median(), pos.max()]
print(stats.round(2))


                         mean  median       max
impressions_90d       5200.37  731.00  517715.0
clicks_90d              16.10    1.00    4178.0
ctr                      0.51    0.07     100.0
engagement_rate          2.53    0.00     100.0
scroll_rate             18.21    5.00     300.0
avg_position (pos>0)    17.03   11.40     245.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal #1 — Claim: pages with more visibility (impressions) earn a better CTR. Test: weighted CTR by impression_tier. Verdict: FALSE — no consistent pattern (low 0.339% → moderate 0.226% → good 0.316% → excellent 0.317%).

Signal #2 — Claim: transactional/commercial intent pages earn higher CTR than informational ones. Test: weighted CTR by main_intent. Verdict: CONFIRMED (modest) — transactional 0.358% > commercial 0.315% > informational 0.290%. (navigational, n=46, is below the ~50-row floor — no verdict there.)

Signal #3 — Claim: CTR and on-page engagement move together. Test: Spearman correlation between ctr and engagement_rate, restricted to sessions_90d >= 30. Verdict: CONFIRMED (moderate) — n=7,114, correlation = 0.372.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal #1: visibility vs CTR
tier_order = ["low", "moderate", "good", "excellent"]
t1 = df[df["impression_tier"].isin(tier_order)].groupby("impression_tier").agg(
    n=("content_id", "size"), clicks=("clicks_90d", "sum"), impr=("impressions_90d", "sum")
).reindex(tier_order)
t1["weighted_ctr_pct"] = (t1["clicks"] / t1["impr"] * 100).round(3)
print("Signal #1 — impression_tier vs weighted CTR")
print(t1[["n", "weighted_ctr_pct"]], "\n")

# Signal #2: intent vs CTR
t2 = df.groupby("main_intent", dropna=False).agg(
    n=("content_id", "size"), clicks=("clicks_90d", "sum"), impr=("impressions_90d", "sum")
)
t2["weighted_ctr_pct"] = (t2["clicks"] / t2["impr"] * 100).round(3)
print("Signal #2 — main_intent vs weighted CTR")
print(t2[["n", "weighted_ctr_pct"]], "\n")

# Signal #3: CTR vs engagement
sub = df[df["sessions_90d"] >= 30]
corr = sub["ctr"].corr(sub["engagement_rate"], method="spearman")
print("Signal #3 — CTR vs engagement_rate (Spearman)")
print("n =", len(sub), "| correlation =", round(corr, 3))


Signal #1 — impression_tier vs weighted CTR
                     n  weighted_ctr_pct
impression_tier                         
low              11248             0.339
moderate         10469             0.226
good              7205             0.316
excellent         1078             0.317 

Signal #2 — main_intent vs weighted CTR
                   n  weighted_ctr_pct
main_intent                           
commercial      4612             0.315
informational  17235             0.290
navigational      46             0.402
transactional   5733             0.358
NaN             2374             0.388 

Signal #3 — CTR vs engagement_rate (Spearman)
n = 7114 | correlation = 0.372


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Flag-linked test — needs_ctr_fix. Claim: a page whose CTR sits well below its position tier's typical CTR is a real, flag-worthy anomaly — the assumption behind FlyRank's needs_ctr_fix logic. Test: for each position_tier (excluding avg_position == 0, "no data"), compute the tier's expected CTR two ways — the volume-weighted mean (what a naive flag would use) and the median (what a typical single page actually gets) — then flag any page whose own CTR is below half the tier's weighted mean, and see what share of each tier that catches.
Verdict: MIXED. Position genuinely drives typical CTR — mean CTR steps down cleanly from top_3 to deep. But CTR inside every tier is so skewed that the median page in top_3 and deep gets literally zero clicks, so flagging anything below half the mean catches 50–85% of every tier — that's not finding anomalies, it's finding the majority. The assumption is directionally right; the implementation needs a median benchmark, not a mean one.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pos = df[df["avg_position"] > 0].copy()
pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

tier = pos.groupby("position_tier").agg(clicks=("clicks_90d", "sum"), impr=("impressions_90d", "sum"))
tier["expected_ctr_mean_pct"] = (tier["clicks"] / tier["impr"] * 100)
tier["expected_ctr_median_pct"] = pos.groupby("position_tier")["ctr"].median()
tier = tier.reindex(pos_order)

pos = pos.join(tier["expected_ctr_mean_pct"], on="position_tier")
pos["ctr_gap_flag"] = pos["ctr"] < (0.5 * pos["expected_ctr_mean_pct"])

flag_rate = pos.groupby("position_tier").agg(n=("content_id", "size"), flagged=("ctr_gap_flag", "sum")).reindex(pos_order)
flag_rate["flag_rate_pct"] = (flag_rate["flagged"] / flag_rate["n"] * 100).round(1)

print(tier[["expected_ctr_mean_pct", "expected_ctr_median_pct"]].round(3))
print()
print(flag_rate)


               expected_ctr_mean_pct  expected_ctr_median_pct
position_tier                                                
top_3                          0.489                     0.00
page_1                         0.350                     0.16
striking                       0.347                     0.11
page_3_5                       0.155                     0.03
deep                           0.041                     0.00

                   n  flagged  flag_rate_pct
position_tier                               
top_3           1116      780           69.9
page_1         11814     6201           52.5
striking        7304     4451           60.9
page_3_5        7242     4308           59.5
deep            1319     1119           84.8


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Position is the reliable signal here — it's the right thing to gate a CTR-fix rule on. Raw impression volume isn't: Signal #1 shows CTR doesn't track visibility tier at all, so a rule that scores by raw impressions rewards scale, not an actual CTR problem.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Visibility vs CTR (Signal #1): no consistent pattern —",
      dict(t1["weighted_ctr_pct"].round(3)))
print()
print("CTR vs engagement (Signal #3): Spearman =", round(corr, 3))
print()
print("needs_ctr_fix flag rate using mean benchmark:")
print(flag_rate["flag_rate_pct"])


Visibility vs CTR (Signal #1): no consistent pattern — {'low': np.float64(0.339), 'moderate': np.float64(0.226), 'good': np.float64(0.316), 'excellent': np.float64(0.317)}

CTR vs engagement (Signal #3): Spearman = 0.372

needs_ctr_fix flag rate using mean benchmark:
position_tier
top_3       69.9
page_1      52.5
striking    60.9
page_3_5    59.5
deep        84.8
Name: flag_rate_pct, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.